# XAI training, tuning, and style profiles

This notebook uses `data/processed/final` as the only source of train, validation, and test data.

Workflow:
1. Build numeric features from final response CSV files.
2. Train several model families on the train split.
3. Compare metrics on validation.
4. Tune hyperparameters on validation only.
5. Evaluate the selected model on test once.
6. Explain the selected model with built-in importance, permutation importance, and SHAP.
7. Build style profiles for the dashboard.

Important: `is_paraphrase` is not a prediction target. It stays available for filtering and profile analysis.

In [1]:
from __future__ import annotations

import json
import math
import re
import string
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from llm_behavior_xai.config import (  # noqa: E402
    LLM_RESULTS_TEST_PROMPTS,
    LLM_RESULTS_TRAIN_PROMPTS,
    LLM_RESULTS_VAL_PROMPTS,
    STYLE_PROFILES_REPORTS_DIR,
    XAI_MODELS_DIR,
    XAI_REPORTS_DIR,
)

FINAL_SPLIT_PATHS = {
    "train": LLM_RESULTS_TRAIN_PROMPTS,
    "val": LLM_RESULTS_VAL_PROMPTS,
    "test": LLM_RESULTS_TEST_PROMPTS,
}

TARGET_COLUMNS = ("model_key", "language")
SELECTION_METRIC = "macro_f1"
RANDOM_STATE = 42

MODELS_DIR = XAI_MODELS_DIR
XAI_DIR = XAI_REPORTS_DIR
PROFILES_DIR = STYLE_PROFILES_REPORTS_DIR

for directory in (
    MODELS_DIR,
    XAI_DIR,
    XAI_DIR / "predictions",
    XAI_DIR / "confusion_matrices",
    XAI_DIR / "shap",
    XAI_DIR / "features",
    XAI_DIR / "importance",
    XAI_DIR / "surrogate_trees",
    PROFILES_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model targets: {TARGET_COLUMNS}")

Project root: d:\main\Desktop\xai\LLM-Behavior-XAI
Model targets: ('model_key', 'language')


## Load final train, validation, and test splits

In [2]:
def load_final_splits(split_paths: dict[str, Path]) -> dict[str, pd.DataFrame]:
    return {split: pd.read_csv(path) for split, path in split_paths.items()}


final_splits = load_final_splits(FINAL_SPLIT_PATHS)

split_overview = []
for split, df in final_splits.items():
    split_overview.append(
        {
            "split": split,
            "rows": len(df),
            "models": df["model_key"].nunique(),
            "languages": df["language"].nunique(),
            "paraphrase_values": df["is_paraphrase"].nunique(),
        }
    )

pd.DataFrame(split_overview)

,split,rows,models,languages,paraphrase_values
0,train,1920,4,2,2
1,val,480,4,2,2
2,test,494,4,2,2


## Build numeric training features

The model never sees raw prompt text, raw response text, provider, model id, prompt id, timestamps, or the paraphrase flag as a target. It only sees numeric metadata and numeric text-response style features.

In [3]:
SOURCE_NUMERIC_COLUMNS = (
    "response_length",
    "generated_tokens",
    "perplexity",
    "avg_logprob",
    "sum_logprob",
)

PREDICTION_CONTEXT_COLUMNS = (
    "run_id",
    "prompt_id",
    "category",
    "prompt_column",
    "language",
    "is_paraphrase",
    "model_key",
    "provider",
    "prompt_text",
    "response",
)

WORD_RE = re.compile(r"\b[\w']+\b", flags=re.UNICODE)
SENTENCE_RE = re.compile(r"[^.!?]+[.!?]*", flags=re.UNICODE)
LIST_MARKER_RE = re.compile(r"(?m)^\s*(?:[-*+]|\d+[.)])\s+")
HEADING_RE = re.compile(r"(?m)^\s{0,3}#{1,6}\s+")

FIRST_PERSON_PRONOUNS = {"i", "me", "my", "mine", "we", "us", "our", "ours", "ja", "mnie", "nas", "nam"}
SECOND_PERSON_PRONOUNS = {"you", "your", "yours", "ty", "ci", "ciebie", "tobie", "wy", "was", "wam"}
HEDGE_WORDS = {"maybe", "perhaps", "probably", "possibly", "likely", "might", "could", "may", "moze", "chyba"}
NEGATION_WORDS = {"no", "not", "never", "none", "cannot", "can't", "nie", "nigdy"}


def tokenize_words(text: str) -> list[str]:
    return [word.lower() for word in WORD_RE.findall(text)]


def split_sentences(text: str) -> list[str]:
    sentences = [sentence.strip() for sentence in SENTENCE_RE.findall(text) if sentence.strip()]
    return sentences if sentences else ([text.strip()] if text.strip() else [])


def safe_divide(numerator: float, denominator: float) -> float:
    return 0.0 if denominator == 0 else float(numerator / denominator)


def mean(values: list[int] | list[float]) -> float:
    return 0.0 if not values else float(np.mean(values))


def count_terms(words: list[str], terms: set[str]) -> int:
    return sum(1 for word in words if word in terms)


def repeated_ngram_ratio(words: list[str], ngram_size: int) -> float:
    if len(words) < ngram_size:
        return 0.0
    ngrams = list(zip(*(words[index:] for index in range(ngram_size))))
    counts = Counter(ngrams)
    repeated = sum(count - 1 for count in counts.values() if count > 1)
    return safe_divide(repeated, len(ngrams))


def word_entropy(word_counts: Counter[str], word_count: int) -> float:
    if word_count == 0:
        return 0.0
    probabilities = [count / word_count for count in word_counts.values()]
    return float(-sum(probability * math.log2(probability) for probability in probabilities))


def count_paragraphs(text: str) -> int:
    return len([paragraph for paragraph in re.split(r"\n\s*\n", text.strip()) if paragraph.strip()])


def extract_response_text_features(text: object) -> dict[str, float]:
    response = "" if pd.isna(text) else str(text)
    words = tokenize_words(response)
    word_counts = Counter(words)
    sentences = split_sentences(response)
    sentence_char_lengths = [len(sentence) for sentence in sentences]
    sentence_word_lengths = [len(tokenize_words(sentence)) for sentence in sentences]
    punctuation_count = sum(1 for char in response if char in string.punctuation)
    uppercase_words = [word for word in WORD_RE.findall(response) if len(word) > 1 and word.isupper()]
    char_count = len(response)
    word_count = len(words)
    unique_word_count = len(word_counts)

    return {
        "text_char_count": float(char_count),
        "text_word_count": float(word_count),
        "text_unique_word_count": float(unique_word_count),
        "text_sentence_count": float(len(sentences)),
        "text_paragraph_count": float(count_paragraphs(response)),
        "text_newline_count": float(response.count("\n")),
        "text_avg_word_length": mean([len(word) for word in words]),
        "text_avg_sentence_words": mean(sentence_word_lengths),
        "text_avg_sentence_chars": mean(sentence_char_lengths),
        "text_type_token_ratio": safe_divide(unique_word_count, word_count),
        "text_hapax_ratio": safe_divide(sum(1 for count in word_counts.values() if count == 1), word_count),
        "text_entropy": word_entropy(word_counts, word_count),
        "text_repetition_rate": 1.0 - safe_divide(unique_word_count, word_count),
        "text_avg_word_frequency": mean(list(word_counts.values())),
        "text_max_word_frequency": float(max(word_counts.values(), default=0)),
        "text_repeated_bigram_ratio": repeated_ngram_ratio(words, 2),
        "text_repeated_trigram_ratio": repeated_ngram_ratio(words, 3),
        "text_punctuation_count": float(punctuation_count),
        "text_punctuation_density": safe_divide(punctuation_count, char_count),
        "text_comma_density": safe_divide(response.count(","), char_count),
        "text_colon_density": safe_divide(response.count(":"), char_count),
        "text_semicolon_density": safe_divide(response.count(";"), char_count),
        "text_question_mark_density": safe_divide(response.count("?"), char_count),
        "text_exclamation_mark_density": safe_divide(response.count("!"), char_count),
        "text_digit_density": safe_divide(sum(char.isdigit() for char in response), char_count),
        "text_uppercase_word_ratio": safe_divide(len(uppercase_words), word_count),
        "text_list_marker_count": float(len(LIST_MARKER_RE.findall(response))),
        "text_markdown_heading_count": float(len(HEADING_RE.findall(response))),
        "text_markdown_bold_count": float(response.count("**") // 2),
        "text_code_fence_count": float(response.count("```") // 2),
        "text_first_person_pronoun_count": float(count_terms(words, FIRST_PERSON_PRONOUNS)),
        "text_first_person_pronoun_density": safe_divide(count_terms(words, FIRST_PERSON_PRONOUNS), word_count),
        "text_second_person_pronoun_count": float(count_terms(words, SECOND_PERSON_PRONOUNS)),
        "text_second_person_pronoun_density": safe_divide(count_terms(words, SECOND_PERSON_PRONOUNS), word_count),
        "text_hedge_word_count": float(count_terms(words, HEDGE_WORDS)),
        "text_hedge_word_density": safe_divide(count_terms(words, HEDGE_WORDS), word_count),
        "text_negation_word_count": float(count_terms(words, NEGATION_WORDS)),
        "text_negation_word_density": safe_divide(count_terms(words, NEGATION_WORDS), word_count),
    }


def build_feature_frame(final_results_df: pd.DataFrame) -> pd.DataFrame:
    responses = final_results_df.get("response", pd.Series([""] * len(final_results_df))).fillna("")
    text_features = pd.DataFrame.from_records(
        [extract_response_text_features(response) for response in responses],
        index=final_results_df.index,
    )
    source_features = pd.DataFrame(index=final_results_df.index)
    for column in SOURCE_NUMERIC_COLUMNS:
        if column in final_results_df.columns:
            source_features[f"source_{column}"] = pd.to_numeric(final_results_df[column], errors="coerce")
    features = pd.concat([source_features, text_features], axis=1)
    return features.replace([np.inf, -np.inf], np.nan).astype(float)


def build_feature_splits(final_splits: dict[str, pd.DataFrame]):
    raw_features = {split: build_feature_frame(df) for split, df in final_splits.items()}
    feature_columns = sorted(raw_features["train"].columns)
    fill_values = raw_features["train"].reindex(columns=feature_columns).median(numeric_only=True).fillna(0.0)
    feature_splits = {}
    for split, features in raw_features.items():
        clean = features.reindex(columns=feature_columns).replace([np.inf, -np.inf], np.nan)
        feature_splits[split] = clean.fillna(fill_values).fillna(0.0).astype(float)
    return feature_splits, feature_columns, fill_values


feature_splits, feature_columns, fill_values = build_feature_splits(final_splits)
for split, features in feature_splits.items():
    features.to_csv(XAI_DIR / "features" / f"{split}_features.csv", index=False)

feature_list = pd.DataFrame({"feature": feature_columns})
feature_list.to_csv(XAI_DIR / "features" / "feature_list.csv", index=False)
with (XAI_DIR / "features" / "feature_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "feature_columns": feature_columns,
            "fill_values": fill_values.to_dict(),
            "source": "data/processed/final",
            "targets": TARGET_COLUMNS,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Feature count: {len(feature_columns)}")
feature_list

Feature count: 43


,feature
0,source_avg_logprob
1,source_generated_tokens
2,source_perplexity
3,source_response_length
4,source_sum_logprob
5,text_avg_sentence_chars
6,text_avg_sentence_words
7,text_avg_word_frequency
8,text_avg_word_length
9,text_char_count


In [4]:
# Short dictionary of all training features.
# `source_*` features come directly from numeric columns in data/processed/final CSVs.
# `text_*` features are calculated from the CSV `response` column.
FEATURE_DESCRIPTIONS = {
    "source_avg_logprob": "CSV value: average token log probability, if available; confidence/fluency proxy.",
    "source_generated_tokens": "CSV value: number of generated tokens, if available; response length proxy.",
    "source_perplexity": "CSV value: perplexity, if available; generation uncertainty proxy.",
    "source_response_length": "CSV value: response length in characters.",
    "source_sum_logprob": "CSV value: total token log probability, if available; confidence plus length proxy.",
    "text_avg_sentence_chars": "Average sentence length in characters.",
    "text_avg_sentence_words": "Average sentence length in words.",
    "text_avg_word_frequency": "Average repetition count per used word.",
    "text_avg_word_length": "Average word length in characters.",
    "text_char_count": "Response length in characters recalculated from response text.",
    "text_code_fence_count": "Number of fenced code blocks marked with triple backticks.",
    "text_colon_density": "Colons per character; often marks lists, explanations, or definitions.",
    "text_comma_density": "Commas per character; rough punctuation/complexity indicator.",
    "text_digit_density": "Digits per character; shows numerical/detail-heavy answers.",
    "text_entropy": "Lexical entropy; higher means word usage is more diverse/uniform.",
    "text_exclamation_mark_density": "Exclamation marks per character; emphasis/expressiveness proxy.",
    "text_first_person_pronoun_count": "Count of first-person words like I/we/ja/my.",
    "text_first_person_pronoun_density": "First-person words divided by total words.",
    "text_hapax_ratio": "Share of words appearing only once; lexical variety proxy.",
    "text_hedge_word_count": "Count of uncertainty words like maybe/could/chyba.",
    "text_hedge_word_density": "Uncertainty words divided by total words.",
    "text_list_marker_count": "Number of bullet or numbered-list markers.",
    "text_markdown_bold_count": "Number of bold markdown spans marked with **.",
    "text_markdown_heading_count": "Number of markdown headings marked with #.",
    "text_max_word_frequency": "Highest repetition count of a single word.",
    "text_negation_word_count": "Count of negation words like not/never/nie.",
    "text_negation_word_density": "Negation words divided by total words.",
    "text_newline_count": "Number of line breaks; formatting/structure proxy.",
    "text_paragraph_count": "Number of paragraph blocks.",
    "text_punctuation_count": "Total punctuation marks.",
    "text_punctuation_density": "Punctuation marks per character.",
    "text_question_mark_density": "Question marks per character.",
    "text_repeated_bigram_ratio": "Share of repeated two-word phrases.",
    "text_repeated_trigram_ratio": "Share of repeated three-word phrases.",
    "text_repetition_rate": "One minus type-token ratio; higher means more repeated vocabulary.",
    "text_second_person_pronoun_count": "Count of second-person words like you/your/ty/wy.",
    "text_second_person_pronoun_density": "Second-person words divided by total words.",
    "text_semicolon_density": "Semicolons per character; complex punctuation indicator.",
    "text_sentence_count": "Number of detected sentences.",
    "text_type_token_ratio": "Unique words divided by total words; lexical diversity proxy.",
    "text_unique_word_count": "Number of distinct words.",
    "text_uppercase_word_ratio": "Uppercase words divided by total words; emphasis/acronym proxy.",
    "text_word_count": "Number of words in the response.",
}

feature_meanings = pd.DataFrame(
    {
        "feature": feature_columns,
        "feature_source": [
            "direct CSV numeric column" if feature.startswith("source_") else "engineered from CSV response"
            for feature in feature_columns
        ],
        "what_it_shows": [FEATURE_DESCRIPTIONS.get(feature, "No description yet.") for feature in feature_columns],
    }
)
feature_meanings.to_csv(XAI_DIR / "features" / "feature_descriptions.csv", index=False)
feature_meanings

,feature,feature_source,what_it_shows
0,source_avg_logprob,direct CSV numeric column,"CSV value: average token log probability, if a..."
1,source_generated_tokens,direct CSV numeric column,"CSV value: number of generated tokens, if avai..."
2,source_perplexity,direct CSV numeric column,"CSV value: perplexity, if available; generatio..."
3,source_response_length,direct CSV numeric column,CSV value: response length in characters.
4,source_sum_logprob,direct CSV numeric column,"CSV value: total token log probability, if ava..."
5,text_avg_sentence_chars,engineered from CSV response,Average sentence length in characters.
6,text_avg_sentence_words,engineered from CSV response,Average sentence length in words.
7,text_avg_word_frequency,engineered from CSV response,Average repetition count per used word.
8,text_avg_word_length,engineered from CSV response,Average word length in characters.
9,text_char_count,engineered from CSV response,Response length in characters recalculated fro...


In [5]:
# Check whether any CSV-derived training feature is almost a direct model identifier.
# High values here do not automatically mean a feature is invalid, but they should be reviewed.
from sklearn.feature_selection import mutual_info_classif


def eta_squared_by_group(values: pd.Series, labels: pd.Series) -> float:
    overall_mean = values.mean()
    total_variation = ((values - overall_mean) ** 2).sum()
    if total_variation == 0 or pd.isna(total_variation):
        return 0.0
    between_variation = 0.0
    for _, group_values in values.groupby(labels):
        between_variation += len(group_values) * (group_values.mean() - overall_mean) ** 2
    return float(between_variation / total_variation)


train_model_labels = final_splits["train"]["model_key"].astype(str)
try:
    mutual_information = mutual_info_classif(
        feature_splits["train"],
        train_model_labels,
        discrete_features=False,
        random_state=RANDOM_STATE,
    )
except Exception as exc:
    print(f"Mutual information check failed: {exc}")
    mutual_information = np.zeros(len(feature_columns))

feature_signal_rows = []
for feature, mi_score in zip(feature_columns, mutual_information):
    values = feature_splits["train"][feature]
    overall_mean = values.mean()
    overall_std = values.std(ddof=0)
    per_model_means = values.groupby(train_model_labels).mean().to_dict()
    if overall_std == 0 or pd.isna(overall_std):
        max_abs_model_mean_z = 0.0
    else:
        max_abs_model_mean_z = max(abs((mean_value - overall_mean) / overall_std) for mean_value in per_model_means.values())
    eta_squared = eta_squared_by_group(values, train_model_labels)
    if eta_squared >= 0.80 or max_abs_model_mean_z >= 2.5:
        flag = "very_strong_model_signal"
    elif eta_squared >= 0.50 or max_abs_model_mean_z >= 1.5:
        flag = "strong_model_signal"
    else:
        flag = ""
    feature_signal_rows.append(
        {
            "feature": feature,
            "eta_squared_model_key": eta_squared,
            "mutual_information_model_key": float(mi_score),
            "max_abs_model_mean_z": float(max_abs_model_mean_z),
            "unique_values": int(values.nunique()),
            "per_model_means": json.dumps({str(key): float(value) for key, value in per_model_means.items()}),
            "review_flag": flag,
        }
    )

feature_model_signal_check = pd.DataFrame(feature_signal_rows).sort_values(
    ["eta_squared_model_key", "mutual_information_model_key"],
    ascending=False,
)
feature_model_signal_check.to_csv(XAI_DIR / "features" / "feature_model_signal_check.csv", index=False)

feature_model_signal_check.head(25)

,feature,eta_squared_model_key,mutual_information_model_key,max_abs_model_mean_z,unique_values,per_model_means,review_flag
23,text_markdown_heading_count,0.719915,0.449718,1.469228,15,"{""gemini-flash-latest"": 5.295833333333333, ""ll...",strong_model_signal
0,source_avg_logprob,0.570425,0.899265,1.245446,961,"{""gemini-flash-latest"": -0.1561415, ""llama-3.1...",strong_model_signal
2,source_perplexity,0.561104,0.901437,1.245486,961,"{""gemini-flash-latest"": 1.1689919999999998, ""l...",strong_model_signal
30,text_punctuation_density,0.522077,0.506477,1.102147,1862,"{""gemini-flash-latest"": 0.06684792595394165, ""...",strong_model_signal
22,text_markdown_bold_count,0.465818,0.397097,1.031938,47,"{""gemini-flash-latest"": 17.18125, ""llama-3.1-8...",
1,source_generated_tokens,0.423370,0.751731,1.036337,593,"{""gemini-flash-latest"": 0.0, ""llama-3.1-8b-gro...",
27,text_newline_count,0.326101,0.240844,0.922870,90,"{""gemini-flash-latest"": 40.16041666666667, ""ll...",
29,text_punctuation_count,0.317795,0.324531,0.938839,403,"{""gemini-flash-latest"": 234.21458333333334, ""l...",
28,text_paragraph_count,0.310933,0.188401,0.895154,29,"{""gemini-flash-latest"": 12.74375, ""llama-3.1-8...",
18,text_hapax_ratio,0.280930,0.369565,0.798729,1776,"{""gemini-flash-latest"": 0.47940535527819617, ""...",


## Check model-specific missing or zero features

In [6]:
# This check uses raw, unfilled feature values.
# It catches suspicious cases where one model has mostly 0/NaN for a feature,
# while other models have regular non-zero values.
raw_feature_frames = []
raw_context_frames = []

for split, source_df in final_splits.items():
    raw_features = build_feature_frame(source_df).reindex(columns=feature_columns)
    raw_features["split"] = split
    raw_feature_frames.append(raw_features)

    context = source_df[["model_key"]].copy()
    context["split"] = split
    raw_context_frames.append(context)

raw_all_features = pd.concat(raw_feature_frames, ignore_index=True)
raw_all_context = pd.concat(raw_context_frames, ignore_index=True)
all_model_labels = raw_all_context["model_key"].astype(str)

zero_nan_rows = []
for feature in feature_columns:
    values = raw_all_features[feature]
    for model_key in sorted(all_model_labels.unique()):
        model_mask = all_model_labels == model_key
        model_values = values[model_mask]
        other_values = values[~model_mask]

        model_nan_ratio = float(model_values.isna().mean())
        other_nan_ratio = float(other_values.isna().mean())
        model_zero_ratio = float((model_values.fillna(np.nan) == 0).mean())
        other_zero_ratio = float((other_values.fillna(np.nan) == 0).mean())
        model_zero_or_nan_ratio = float((model_values.isna() | (model_values == 0)).mean())
        other_zero_or_nan_ratio = float((other_values.isna() | (other_values == 0)).mean())

        model_normal_values = model_values.dropna()
        model_normal_values = model_normal_values[model_normal_values != 0]
        other_normal_values = other_values.dropna()
        other_normal_values = other_normal_values[other_normal_values != 0]

        difference = model_zero_or_nan_ratio - other_zero_or_nan_ratio
        if model_zero_or_nan_ratio >= 0.80 and other_zero_or_nan_ratio <= 0.30:
            flag = "model_mostly_zero_or_nan"
        elif difference >= 0.50:
            flag = "model_more_zero_or_nan"
        elif difference <= -0.50:
            flag = "others_more_zero_or_nan"
        else:
            flag = ""

        zero_nan_rows.append(
            {
                "feature": feature,
                "model_key": model_key,
                "model_zero_or_nan_ratio": model_zero_or_nan_ratio,
                "other_zero_or_nan_ratio": other_zero_or_nan_ratio,
                "difference": difference,
                "model_nan_ratio": model_nan_ratio,
                "other_nan_ratio": other_nan_ratio,
                "model_zero_ratio": model_zero_ratio,
                "other_zero_ratio": other_zero_ratio,
                "model_nonzero_nonnull_mean": float(model_normal_values.mean()) if len(model_normal_values) else np.nan,
                "other_nonzero_nonnull_mean": float(other_normal_values.mean()) if len(other_normal_values) else np.nan,
                "review_flag": flag,
            }
        )

feature_zero_nan_check = pd.DataFrame(zero_nan_rows).sort_values(
    ["review_flag", "difference"],
    ascending=[False, False],
)
feature_zero_nan_check.to_csv(XAI_DIR / "features" / "feature_zero_nan_by_model_check.csv", index=False)

excluded_zero_nan_features = sorted(
    feature_zero_nan_check.loc[feature_zero_nan_check["review_flag"] != "", "feature"].unique()
)
pd.DataFrame({"excluded_feature": excluded_zero_nan_features}).to_csv(
    XAI_DIR / "features" / "excluded_zero_nan_features.csv",
    index=False,
)

if excluded_zero_nan_features:
    print("Excluding features with model-specific zero/NaN patterns:")
    for feature in excluded_zero_nan_features:
        print(f"- {feature}")
else:
    print("No model-specific zero/NaN features were flagged for exclusion.")

# From this point onward, all training uses only the filtered feature set.
feature_columns = [feature for feature in feature_columns if feature not in excluded_zero_nan_features]
feature_splits = {split: features[feature_columns].copy() for split, features in feature_splits.items()}
feature_list = pd.DataFrame({"feature": feature_columns})
feature_list.to_csv(XAI_DIR / "features" / "feature_list.csv", index=False)

if "feature_meanings" in globals():
    feature_meanings = feature_meanings.copy()
    feature_meanings["used_for_training"] = feature_meanings["feature"].isin(feature_columns)
    feature_meanings["excluded_reason"] = np.where(
        feature_meanings["feature"].isin(excluded_zero_nan_features),
        "model-specific zero/NaN pattern",
        "",
    )
    feature_meanings.to_csv(XAI_DIR / "features" / "feature_descriptions.csv", index=False)

with (XAI_DIR / "features" / "feature_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "feature_columns": feature_columns,
            "excluded_zero_nan_features": excluded_zero_nan_features,
            "source": "data/processed/final",
            "targets": TARGET_COLUMNS,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Remaining training feature count: {len(feature_columns)}")
feature_zero_nan_check[feature_zero_nan_check["review_flag"] != ""].head(30)

Excluding features with model-specific zero/NaN patterns:
- source_avg_logprob
- source_generated_tokens
- source_perplexity
- source_sum_logprob
- text_markdown_bold_count
- text_markdown_heading_count
- text_question_mark_density
Remaining training feature count: 36


,feature,model_key,model_zero_or_nan_ratio,other_zero_or_nan_ratio,difference,model_nan_ratio,other_nan_ratio,model_zero_ratio,other_zero_ratio,model_nonzero_nonnull_mean,other_nonzero_nonnull_mean,review_flag
124,text_question_mark_density,gemini-flash-latest,0.356948,0.912037,-0.555089,0.0,0.000000,0.356948,0.912037,0.000909,0.001585,others_more_zero_or_nan
88,text_markdown_bold_count,gemini-flash-latest,0.002725,0.605093,-0.602368,0.0,0.000000,0.002725,0.605093,16.916667,10.166471,others_more_zero_or_nan
2,source_avg_logprob,mistral-7b-hf,0.000000,0.668813,-0.668813,0.0,0.668813,0.000000,0.000000,-0.120907,-0.243944,others_more_zero_or_nan
3,source_avg_logprob,phi-3-mini-hf,0.000000,0.668813,-0.668813,0.0,0.668813,0.000000,0.000000,-0.243944,-0.120907,others_more_zero_or_nan
6,source_generated_tokens,mistral-7b-hf,0.000000,0.668813,-0.668813,0.0,0.000000,0.000000,0.668813,481.263889,1245.204167,others_more_zero_or_nan
7,source_generated_tokens,phi-3-mini-hf,0.000000,0.668813,-0.668813,0.0,0.000000,0.000000,0.668813,1245.204167,481.263889,others_more_zero_or_nan
10,source_perplexity,mistral-7b-hf,0.000000,0.668813,-0.668813,0.0,0.668813,0.000000,0.000000,1.128981,1.280189,others_more_zero_or_nan
11,source_perplexity,phi-3-mini-hf,0.000000,0.668813,-0.668813,0.0,0.668813,0.000000,0.000000,1.280189,1.128981,others_more_zero_or_nan
18,source_sum_logprob,mistral-7b-hf,0.000000,0.668813,-0.668813,0.0,0.668813,0.000000,0.000000,-56.790103,-326.658963,others_more_zero_or_nan
19,source_sum_logprob,phi-3-mini-hf,0.000000,0.668813,-0.668813,0.0,0.668813,0.000000,0.000000,-326.658963,-56.790103,others_more_zero_or_nan


## Train candidate model families and compare validation metrics

In [7]:
def normalize_label(value: object) -> str:
    if pd.isna(value):
        return "missing"
    if isinstance(value, (bool, np.bool_)):
        return str(bool(value))
    return str(value)


def prediction_context(df: pd.DataFrame) -> pd.DataFrame:
    columns = [column for column in PREDICTION_CONTEXT_COLUMNS if column in df.columns]
    return df[columns].copy()


def safe_filename_part(value: str) -> str:
    return "".join(char if char.isalnum() else "_" for char in value).strip("_").lower() or "class"


def encode_target(target: str):
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(final_splits["train"][target].map(normalize_label))
    y_val = encoder.transform(final_splits["val"][target].map(normalize_label))
    y_test = encoder.transform(final_splits["test"][target].map(normalize_label))
    return encoder, {"train": y_train, "val": y_val, "test": y_test}


def metric_row(target: str, split: str, model_name: str, params: dict[str, Any], y_true, y_pred, class_names):
    labels = list(range(len(class_names)))
    return {
        "target": target,
        "split": split,
        "model_name": model_name,
        "params": json.dumps(params, sort_keys=True),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "classification_report": json.dumps(
            classification_report(
                y_true,
                y_pred,
                labels=labels,
                target_names=class_names,
                output_dict=True,
                zero_division=0,
            ),
            ensure_ascii=False,
        ),
    }


def build_model_candidates() -> dict[str, tuple[Any, dict[str, list[Any]]]]:
    return {
        "logistic_regression": (
            Pipeline(
                steps=[
                    ("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=RANDOM_STATE)),
                ]
            ),
            {
                "model__C": [0.1, 1.0, 5.0],
                "model__solver": ["lbfgs"],
            },
        ),
        "random_forest": (
            RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
            {
                "n_estimators": [200, 500],
                "max_depth": [None, 8, 16],
                "min_samples_leaf": [1, 3],
            },
        ),
        "extra_trees": (
            ExtraTreesClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
            {
                "n_estimators": [200, 500],
                "max_depth": [None, 8, 16],
                "min_samples_leaf": [1, 3],
            },
        ),
        "gradient_boosting": (
            GradientBoostingClassifier(random_state=RANDOM_STATE),
            {
                "n_estimators": [100, 200],
                "learning_rate": [0.05, 0.1],
                "max_depth": [2, 3],
            },
        ),
    }


def fit_with_params(estimator, params: dict[str, Any]):
    model = clone(estimator)
    model.set_params(**params)
    return model


validation_rows = []
best_by_target = {}

for target in TARGET_COLUMNS:
    encoder, encoded_targets = encode_target(target)
    class_names = list(encoder.classes_)
    target_candidates = []

    for model_name, (base_estimator, param_grid) in build_model_candidates().items():
        for params in ParameterGrid(param_grid):
            model = fit_with_params(base_estimator, params)
            model.fit(feature_splits["train"], encoded_targets["train"])
            val_pred = model.predict(feature_splits["val"])
            row = metric_row(target, "val", model_name, params, encoded_targets["val"], val_pred, class_names)
            validation_rows.append(row)
            target_candidates.append((row[SELECTION_METRIC], row["balanced_accuracy"], model_name, params, model))

    target_candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
    best_score, best_balanced_acc, best_model_name, best_params, best_model = target_candidates[0]
    best_by_target[target] = {
        "encoder": encoder,
        "encoded_targets": encoded_targets,
        "class_names": class_names,
        "model_name": best_model_name,
        "params": best_params,
        "model": best_model,
        "val_score": best_score,
        "val_balanced_accuracy": best_balanced_acc,
    }

validation_metrics = pd.DataFrame(validation_rows).sort_values(
    ["target", SELECTION_METRIC, "balanced_accuracy"],
    ascending=[True, False, False],
)
validation_metrics.to_csv(XAI_DIR / "validation_tuning_metrics.csv", index=False)
validation_metrics.head(20)

,target,split,model_name,params,accuracy,macro_f1,balanced_accuracy,classification_report
40,language,val,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 3, ""n_...",0.981250,0.981246,0.981250,"{""en"": {""precision"": 0.9676113360323887, ""reca..."
48,language,val,random_forest,"{""max_depth"": 16, ""min_samples_leaf"": 3, ""n_es...",0.981250,0.981246,0.981250,"{""en"": {""precision"": 0.9676113360323887, ""reca..."
41,language,val,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 3, ""n_...",0.979167,0.979161,0.979167,"{""en"": {""precision"": 0.9637096774193549, ""reca..."
49,language,val,random_forest,"{""max_depth"": 16, ""min_samples_leaf"": 3, ""n_es...",0.979167,0.979161,0.979167,"{""en"": {""precision"": 0.9637096774193549, ""reca..."
39,language,val,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.977083,0.977078,0.977083,"{""en"": {""precision"": 0.9635627530364372, ""reca..."
42,language,val,random_forest,"{""max_depth"": 8, ""min_samples_leaf"": 1, ""n_est...",0.977083,0.977078,0.977083,"{""en"": {""precision"": 0.9635627530364372, ""reca..."
43,language,val,random_forest,"{""max_depth"": 8, ""min_samples_leaf"": 1, ""n_est...",0.977083,0.977078,0.977083,"{""en"": {""precision"": 0.9635627530364372, ""reca..."
45,language,val,random_forest,"{""max_depth"": 8, ""min_samples_leaf"": 3, ""n_est...",0.977083,0.977078,0.977083,"{""en"": {""precision"": 0.9635627530364372, ""reca..."
47,language,val,random_forest,"{""max_depth"": 16, ""min_samples_leaf"": 1, ""n_es...",0.977083,0.977078,0.977083,"{""en"": {""precision"": 0.9635627530364372, ""reca..."
44,language,val,random_forest,"{""max_depth"": 8, ""min_samples_leaf"": 3, ""n_est...",0.977083,0.977075,0.977083,"{""en"": {""precision"": 0.9598393574297188, ""reca..."


## Selected hyperparameters from validation

In [8]:
selected_rows = []
for target, bundle in best_by_target.items():
    selected_rows.append(
        {
            "target": target,
            "selected_model": bundle["model_name"],
            "selected_params": json.dumps(bundle["params"], sort_keys=True),
            "validation_macro_f1": bundle["val_score"],
            "validation_balanced_accuracy": bundle["val_balanced_accuracy"],
        }
    )

selected_models = pd.DataFrame(selected_rows)
selected_models.to_csv(XAI_DIR / "selected_models.csv", index=False)
selected_models

,target,selected_model,selected_params,validation_macro_f1,validation_balanced_accuracy
0,model_key,extra_trees,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_...",0.910093,0.910417
1,language,random_forest,"{""max_depth"": null, ""min_samples_leaf"": 3, ""n_...",0.981246,0.981250


## Final test metrics for selected models

The test split is evaluated here after model selection. Do not use these scores to tune again.

In [9]:
def save_predictions(target: str, split: str, source_df: pd.DataFrame, y_true, y_pred, probabilities, class_names):
    prediction_df = prediction_context(source_df)
    prediction_df["target"] = target
    prediction_df["split"] = split
    prediction_df["y_true"] = best_by_target[target]["encoder"].inverse_transform(y_true)
    prediction_df["y_pred"] = best_by_target[target]["encoder"].inverse_transform(y_pred)
    prediction_df["correct"] = prediction_df["y_true"] == prediction_df["y_pred"]
    prediction_df["prediction_confidence"] = probabilities.max(axis=1)
    for class_index, class_name in enumerate(class_names):
        prediction_df[f"probability_{safe_filename_part(class_name)}"] = probabilities[:, class_index]
    prediction_df.to_csv(XAI_DIR / "predictions" / f"{target}_{split}_predictions.csv", index=False)


def save_confusion_matrix(target: str, split: str, y_true, y_pred, class_names):
    labels = list(range(len(class_names)))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    pd.DataFrame(matrix, index=class_names, columns=class_names).to_csv(
        XAI_DIR / "confusion_matrices" / f"{target}_{split}_confusion_matrix.csv"
    )
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(f"{target} confusion matrix ({split})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(labels, labels=class_names, rotation=45, ha="right")
    ax.set_yticks(labels, labels=class_names)
    for row_index in labels:
        for column_index in labels:
            ax.text(column_index, row_index, matrix[row_index, column_index], ha="center", va="center")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(XAI_DIR / "confusion_matrices" / f"{target}_{split}_confusion_matrix.png")
    plt.close(fig)


all_metrics = []
for target, bundle in best_by_target.items():
    model = bundle["model"]
    class_names = bundle["class_names"]
    for split in ("train", "val", "test"):
        y_true = bundle["encoded_targets"][split]
        y_pred = model.predict(feature_splits[split])
        probabilities = model.predict_proba(feature_splits[split])
        all_metrics.append(metric_row(target, split, bundle["model_name"], bundle["params"], y_true, y_pred, class_names))
        save_predictions(target, split, final_splits[split], y_true, y_pred, probabilities, class_names)
        save_confusion_matrix(target, split, y_true, y_pred, class_names)

    joblib.dump(model, MODELS_DIR / f"{target}_best_model.joblib")
    with (MODELS_DIR / f"{target}_metadata.json").open("w", encoding="utf-8") as file:
        json.dump(
            {
                "target": target,
                "model_name": bundle["model_name"],
                "params": bundle["params"],
                "class_names": class_names,
                "feature_columns": feature_columns,
                "selection_metric": SELECTION_METRIC,
                "selection_split": "val",
            },
            file,
            ensure_ascii=False,
            indent=2,
        )

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(XAI_DIR / "all_metrics.csv", index=False)
metrics_df[metrics_df["split"].isin(["val", "test"])][
    ["target", "split", "model_name", "accuracy", "macro_f1", "balanced_accuracy", "params"]
]

,target,split,model_name,accuracy,macro_f1,balanced_accuracy,params
1,model_key,val,extra_trees,0.910417,0.910093,0.910417,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_..."
2,model_key,test,extra_trees,0.894737,0.892045,0.892320,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_..."
4,language,val,random_forest,0.981250,0.981246,0.981250,"{""max_depth"": null, ""min_samples_leaf"": 3, ""n_..."
5,language,test,random_forest,0.981781,0.981775,0.981724,"{""max_depth"": null, ""min_samples_leaf"": 3, ""n_..."


## Explainability: built-in importance, permutation importance, and SHAP

In [10]:
def estimator_from_model(model):
    return model.named_steps["model"] if isinstance(model, Pipeline) else model


def built_in_importance(target: str, model) -> pd.DataFrame:
    estimator = estimator_from_model(model)
    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        kind = "feature_importances"
    elif hasattr(estimator, "coef_"):
        values = np.abs(estimator.coef_).mean(axis=0)
        kind = "absolute_coefficients"
    else:
        values = np.zeros(len(feature_columns))
        kind = "not_available"
    return pd.DataFrame(
        {
            "target": target,
            "feature": feature_columns,
            "importance": values,
            "importance_kind": kind,
        }
    ).sort_values("importance", ascending=False)


def permutation_importance_frame(target: str, model) -> pd.DataFrame:
    bundle = best_by_target[target]
    result = permutation_importance(
        model,
        feature_splits["val"],
        bundle["encoded_targets"]["val"],
        scoring="f1_macro",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return pd.DataFrame(
        {
            "target": target,
            "feature": feature_columns,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
        }
    ).sort_values("importance_mean", ascending=False)


def shap_values_to_importance(target: str, values: np.ndarray, class_names: list[str]) -> pd.DataFrame:
    records = []
    if values.ndim == 2:
        values = values[:, :, None]
    for class_index, class_name in enumerate(class_names[: values.shape[2]]):
        mean_abs = np.abs(values[:, :, class_index]).mean(axis=0)
        for rank, feature_index in enumerate(np.argsort(mean_abs)[::-1], start=1):
            records.append(
                {
                    "target": target,
                    "class_name": class_name,
                    "feature": feature_columns[feature_index],
                    "mean_abs_shap": mean_abs[feature_index],
                    "rank": rank,
                    "importance_kind": "shap",
                    "note": "",
                }
            )
    overall = np.abs(values).mean(axis=(0, 2))
    for rank, feature_index in enumerate(np.argsort(overall)[::-1], start=1):
        records.append(
            {
                "target": target,
                "class_name": "__overall__",
                "feature": feature_columns[feature_index],
                "mean_abs_shap": overall[feature_index],
                "rank": rank,
                "importance_kind": "shap",
                "note": "",
            }
        )
    return pd.DataFrame(records)


def fallback_shap_frame(target: str, model, reason: str) -> pd.DataFrame:
    fallback = built_in_importance(target, model)
    fallback = fallback.rename(columns={"importance": "mean_abs_shap"})
    fallback["class_name"] = "__overall__"
    fallback["rank"] = range(1, len(fallback) + 1)
    fallback["importance_kind"] = "fallback_importance"
    fallback["note"] = reason
    return fallback[["target", "class_name", "feature", "mean_abs_shap", "rank", "importance_kind", "note"]]


def explain_with_shap(target: str, model) -> pd.DataFrame:
    class_names = best_by_target[target]["class_names"]
    background = feature_splits["train"].sample(n=min(100, len(feature_splits["train"])), random_state=RANDOM_STATE)
    sample = feature_splits["val"].sample(n=min(80, len(feature_splits["val"])), random_state=RANDOM_STATE)
    try:
        import shap

        explainer = shap.Explainer(model.predict_proba, background, algorithm="permutation")
        shap_values = explainer(sample, max_evals=2 * len(feature_columns) + 1)
        return shap_values_to_importance(target, np.asarray(shap_values.values), class_names)
    except Exception as exc:
        return fallback_shap_frame(target, model, str(exc))


def plot_top_importance(df: pd.DataFrame, value_column: str, title: str, output_path: Path, top_n: int = 20):
    top = df.sort_values(value_column, ascending=False).head(top_n).sort_values(value_column)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(top["feature"], top[value_column])
    ax.set_title(title)
    ax.set_xlabel(value_column)
    fig.tight_layout()
    fig.savefig(output_path)
    plt.close(fig)


importance_outputs = []
permutation_outputs = []
shap_outputs = []

for target, bundle in best_by_target.items():
    model = bundle["model"]

    built_in = built_in_importance(target, model)
    built_in.to_csv(XAI_DIR / "importance" / f"{target}_built_in_importance.csv", index=False)
    plot_top_importance(
        built_in,
        "importance",
        f"Built-in importance for {target}",
        XAI_DIR / "importance" / f"{target}_built_in_importance.png",
    )
    importance_outputs.append(built_in.assign(method="built_in"))

    permutation_df = permutation_importance_frame(target, model)
    permutation_df.to_csv(XAI_DIR / "importance" / f"{target}_permutation_importance.csv", index=False)
    plot_top_importance(
        permutation_df,
        "importance_mean",
        f"Permutation importance for {target}",
        XAI_DIR / "importance" / f"{target}_permutation_importance.png",
    )
    permutation_outputs.append(permutation_df.assign(method="permutation"))

    shap_df = explain_with_shap(target, model)
    shap_df.to_csv(XAI_DIR / "shap" / f"{target}_shap_importance.csv", index=False)
    overall_shap = shap_df[shap_df["class_name"] == "__overall__"]
    plot_top_importance(
        overall_shap.rename(columns={"mean_abs_shap": "importance"}),
        "importance",
        f"SHAP importance for {target}",
        XAI_DIR / "shap" / f"{target}_shap_importance.png",
    )
    shap_outputs.append(shap_df)

pd.concat(importance_outputs).head(20)

PermutationExplainer explainer: 81it [00:25,  2.45it/s]                        


,target,feature,importance,importance_kind,method
24,model_key,text_punctuation_density,0.075916,feature_importances,built_in
33,model_key,text_unique_word_count,0.056062,feature_importances,built_in
14,model_key,text_hapax_ratio,0.055924,feature_importances,built_in
23,model_key,text_punctuation_count,0.049778,feature_importances,built_in
10,model_key,text_entropy,0.049306,feature_importances,built_in
32,model_key,text_type_token_ratio,0.047187,feature_importances,built_in
27,model_key,text_repetition_rate,0.045333,feature_importances,built_in
25,model_key,text_repeated_bigram_ratio,0.044067,feature_importances,built_in
0,model_key,source_response_length,0.040678,feature_importances,built_in
5,model_key,text_char_count,0.039567,feature_importances,built_in


## Feature group importance

This groups individual features into human-readable behavior categories, then aggregates importance scores per group.

In [11]:
def feature_group(feature: str) -> str:
    if feature in {"source_avg_logprob", "source_sum_logprob", "source_perplexity"}:
        return "generation_confidence"
    if feature in {
        "source_response_length",
        "source_generated_tokens",
        "text_char_count",
        "text_word_count",
        "text_sentence_count",
        "text_paragraph_count",
        "text_newline_count",
        "text_avg_sentence_chars",
        "text_avg_sentence_words",
        "text_avg_word_length",
    }:
        return "length_and_structure"
    if feature in {
        "text_unique_word_count",
        "text_type_token_ratio",
        "text_hapax_ratio",
        "text_entropy",
        "text_repetition_rate",
        "text_avg_word_frequency",
        "text_max_word_frequency",
        "text_repeated_bigram_ratio",
        "text_repeated_trigram_ratio",
    }:
        return "lexical_diversity_and_repetition"
    if feature in {
        "text_punctuation_count",
        "text_punctuation_density",
        "text_comma_density",
        "text_colon_density",
        "text_semicolon_density",
        "text_question_mark_density",
        "text_exclamation_mark_density",
        "text_digit_density",
        "text_uppercase_word_ratio",
        "text_list_marker_count",
        "text_markdown_bold_count",
        "text_markdown_heading_count",
        "text_code_fence_count",
    }:
        return "formatting_and_punctuation"
    if feature in {
        "text_first_person_pronoun_count",
        "text_first_person_pronoun_density",
        "text_second_person_pronoun_count",
        "text_second_person_pronoun_density",
        "text_hedge_word_count",
        "text_hedge_word_density",
        "text_negation_word_count",
        "text_negation_word_density",
    }:
        return "stance_pronouns_and_uncertainty"
    return "other"


def normalize_group_importance(group_df: pd.DataFrame, value_column: str) -> pd.DataFrame:
    group_df = group_df.copy()
    group_df[value_column] = group_df[value_column].clip(lower=0)
    totals = group_df.groupby(["target", "method"])[value_column].transform("sum")
    group_df["importance_share"] = np.where(totals > 0, group_df[value_column] / totals, 0.0)
    return group_df


group_importance_parts = []

built_in_all = pd.concat(importance_outputs, ignore_index=True)
built_in_all["feature_group"] = built_in_all["feature"].map(feature_group)
built_in_groups = (
    built_in_all.groupby(["target", "method", "feature_group"], as_index=False)["importance"].sum()
)
group_importance_parts.append(normalize_group_importance(built_in_groups, "importance"))

permutation_all = pd.concat(permutation_outputs, ignore_index=True)
permutation_all["feature_group"] = permutation_all["feature"].map(feature_group)
permutation_groups = (
    permutation_all.groupby(["target", "method", "feature_group"], as_index=False)["importance_mean"].sum()
    .rename(columns={"importance_mean": "importance"})
)
group_importance_parts.append(normalize_group_importance(permutation_groups, "importance"))

shap_all = pd.concat(shap_outputs, ignore_index=True)
shap_overall = shap_all[shap_all["class_name"] == "__overall__"].copy()
shap_overall["method"] = "shap"
shap_overall["feature_group"] = shap_overall["feature"].map(feature_group)
shap_groups = (
    shap_overall.groupby(["target", "method", "feature_group"], as_index=False)["mean_abs_shap"].sum()
    .rename(columns={"mean_abs_shap": "importance"})
)
group_importance_parts.append(normalize_group_importance(shap_groups, "importance"))

feature_group_importance = pd.concat(group_importance_parts, ignore_index=True)
feature_group_importance.to_csv(XAI_DIR / "importance" / "feature_group_importance.csv", index=False)

for target in TARGET_COLUMNS:
    target_df = feature_group_importance[
        (feature_group_importance["target"] == target) & (feature_group_importance["method"] == "shap")
    ].sort_values("importance_share")
    if not target_df.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(target_df["feature_group"], target_df["importance_share"])
        ax.set_title(f"Feature group importance for {target} (SHAP)")
        ax.set_xlabel("Share of importance")
        fig.tight_layout()
        fig.savefig(XAI_DIR / "importance" / f"{target}_feature_group_importance.png")
        plt.close(fig)

feature_group_importance.sort_values(["target", "method", "importance_share"], ascending=[True, True, False])

,target,method,feature_group,importance,importance_share
3,language,built_in,stance_pronouns_and_uncertainty,0.682088,0.682088
1,language,built_in,length_and_structure,0.147882,0.147882
2,language,built_in,lexical_diversity_and_repetition,0.089476,0.089476
0,language,built_in,formatting_and_punctuation,0.080554,0.080554
11,language,permutation,stance_pronouns_and_uncertainty,0.059213,0.594162
8,language,permutation,formatting_and_punctuation,0.017724,0.177846
9,language,permutation,length_and_structure,0.015008,0.150595
10,language,permutation,lexical_diversity_and_repetition,0.007713,0.077397
19,language,shap,stance_pronouns_and_uncertainty,0.355222,0.656397
17,language,shap,length_and_structure,0.081610,0.150803


## Surrogate decision trees

A small decision tree is trained to imitate the selected model. This gives simple approximate rules for how the trained model behaves.

In [12]:
def train_surrogate_tree(target: str, max_depth: int = 3, min_samples_leaf: int = 30):
    main_model = best_by_target[target]["model"]
    class_names = best_by_target[target]["class_names"]
    pseudo_train = main_model.predict(feature_splits["train"])

    surrogate = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )
    surrogate.fit(feature_splits["train"], pseudo_train)

    rows = []
    for split in ("train", "val", "test"):
        main_predictions = main_model.predict(feature_splits[split])
        surrogate_predictions = surrogate.predict(feature_splits[split])
        true_labels = best_by_target[target]["encoded_targets"][split]
        rows.append(
            {
                "target": target,
                "split": split,
                "surrogate_max_depth": max_depth,
                "surrogate_min_samples_leaf": min_samples_leaf,
                "fidelity_accuracy": accuracy_score(main_predictions, surrogate_predictions),
                "fidelity_macro_f1": f1_score(
                    main_predictions,
                    surrogate_predictions,
                    average="macro",
                    zero_division=0,
                ),
                "surrogate_task_accuracy": accuracy_score(true_labels, surrogate_predictions),
                "surrogate_task_macro_f1": f1_score(
                    true_labels,
                    surrogate_predictions,
                    average="macro",
                    zero_division=0,
                ),
            }
        )

    rules = export_text(surrogate, feature_names=feature_columns, show_weights=True)
    (XAI_DIR / "surrogate_trees" / f"{target}_surrogate_tree_rules.txt").write_text(rules, encoding="utf-8")
    joblib.dump(surrogate, MODELS_DIR / f"{target}_surrogate_tree.joblib")

    fig, ax = plt.subplots(figsize=(22, 10))
    plot_tree(
        surrogate,
        feature_names=feature_columns,
        class_names=class_names,
        filled=True,
        rounded=True,
        fontsize=8,
        ax=ax,
    )
    ax.set_title(f"Surrogate decision tree for {target}")
    fig.tight_layout()
    fig.savefig(XAI_DIR / "surrogate_trees" / f"{target}_surrogate_tree.png")
    plt.close(fig)

    return pd.DataFrame(rows)


surrogate_metrics = pd.concat(
    [train_surrogate_tree(target) for target in TARGET_COLUMNS],
    ignore_index=True,
)
surrogate_metrics.to_csv(XAI_DIR / "surrogate_trees" / "surrogate_tree_metrics.csv", index=False)
surrogate_metrics

,target,split,surrogate_max_depth,surrogate_min_samples_leaf,fidelity_accuracy,fidelity_macro_f1,surrogate_task_accuracy,surrogate_task_macro_f1
0,model_key,train,3,30,0.736458,0.733128,0.736458,0.733128
1,model_key,val,3,30,0.735417,0.733406,0.737500,0.735571
2,model_key,test,3,30,0.740891,0.730479,0.724696,0.715449
3,language,train,3,30,0.956250,0.956136,0.944271,0.944186
4,language,val,3,30,0.954167,0.953936,0.935417,0.935212
5,language,test,3,30,0.945344,0.945068,0.927126,0.926891


## Style profiles by LLM model

These profiles use the same features, but they are descriptive analysis, not another prediction target.

In [13]:
def combine_profile_data() -> pd.DataFrame:
    records = []
    context_columns = ["prompt_id", "category", "language", "is_paraphrase", "model_key", "provider"]
    for split, source_df in final_splits.items():
        available_context = [column for column in context_columns if column in source_df.columns]
        split_df = pd.concat(
            [source_df[available_context].reset_index(drop=True), feature_splits[split].reset_index(drop=True)],
            axis=1,
        )
        split_df["split"] = split
        records.append(split_df)
    return pd.concat(records, ignore_index=True)


def build_model_feature_summary(profile_data: pd.DataFrame) -> pd.DataFrame:
    summary = profile_data.groupby("model_key")[feature_columns].agg(["mean", "median", "std"]).reset_index()
    summary.columns = ["_".join(column).rstrip("_") for column in summary.columns.to_flat_index()]
    return summary


def build_model_effect_sizes(profile_data: pd.DataFrame) -> pd.DataFrame:
    records = []
    for model_key in sorted(profile_data["model_key"].dropna().unique()):
        model_rows = profile_data[profile_data["model_key"] == model_key]
        other_rows = profile_data[profile_data["model_key"] != model_key]
        for feature in feature_columns:
            model_mean = float(model_rows[feature].mean())
            other_mean = float(other_rows[feature].mean())
            model_std = float(model_rows[feature].std(ddof=1))
            other_std = float(other_rows[feature].std(ddof=1))
            std_values = [value**2 for value in (model_std, other_std) if not np.isnan(value)]
            pooled_std = float(np.sqrt(np.mean(std_values))) if std_values else 0.0
            effect_size = 0.0 if pooled_std == 0 else (model_mean - other_mean) / pooled_std
            records.append(
                {
                    "model_key": model_key,
                    "feature": feature,
                    "model_mean": model_mean,
                    "other_models_mean": other_mean,
                    "effect_size": effect_size,
                    "abs_effect_size": abs(effect_size),
                    "direction": "higher" if effect_size >= 0 else "lower",
                }
            )
    return pd.DataFrame(records).sort_values(["model_key", "abs_effect_size"], ascending=[True, False])


def group_mean(rows: pd.DataFrame, group_column: str, group_value: object, feature: str) -> float:
    group_rows = rows[rows[group_column] == group_value]
    return 0.0 if group_rows.empty else float(group_rows[feature].mean())


def build_language_paraphrase_sensitivity(profile_data: pd.DataFrame) -> pd.DataFrame:
    records = []
    for model_key in sorted(profile_data["model_key"].dropna().unique()):
        model_rows = profile_data[profile_data["model_key"] == model_key]
        for feature in feature_columns:
            en_mean = group_mean(model_rows, "language", "en", feature)
            pl_mean = group_mean(model_rows, "language", "pl", feature)
            base_mean = group_mean(model_rows, "is_paraphrase", False, feature)
            paraphrase_mean = group_mean(model_rows, "is_paraphrase", True, feature)
            records.append(
                {
                    "model_key": model_key,
                    "feature": feature,
                    "en_mean": en_mean,
                    "pl_mean": pl_mean,
                    "language_difference_en_minus_pl": en_mean - pl_mean,
                    "base_prompt_mean": base_mean,
                    "paraphrase_mean": paraphrase_mean,
                    "paraphrase_difference_true_minus_false": paraphrase_mean - base_mean,
                }
            )
    return pd.DataFrame(records)


def write_profile_markdown(top_features: pd.DataFrame, shap_summary: pd.DataFrame):
    lines = [
        "# Style Profiles",
        "",
        "Profiles are generated from engineered features derived from data/processed/final responses.",
        "",
    ]
    for model_key in sorted(top_features["model_key"].unique()):
        lines.extend([f"## {model_key}", "", "Top descriptive differences:"])
        model_features = top_features[top_features["model_key"] == model_key]
        for _, row in model_features.iterrows():
            lines.append(
                f"- `{row['feature']}` is {row['direction']} than other models "
                f"(effect size: {row['effect_size']:.3f})."
            )
        model_shap = shap_summary[shap_summary["class_name"] == model_key].sort_values("rank").head(5)
        if not model_shap.empty:
            lines.extend(["", "Top SHAP features for this model-key classifier class:"])
            for _, row in model_shap.iterrows():
                lines.append(f"- `{row['feature']}` (mean absolute SHAP: {row['mean_abs_shap']:.6f}).")
        lines.append("")
    (PROFILES_DIR / "style_profiles.md").write_text("\n".join(lines), encoding="utf-8")


profile_data = combine_profile_data()
feature_summary = build_model_feature_summary(profile_data)
effect_sizes = build_model_effect_sizes(profile_data)
top_features = effect_sizes.groupby("model_key", group_keys=False).head(8).reset_index(drop=True)
sensitivity = build_language_paraphrase_sensitivity(profile_data)

model_key_shap_path = XAI_DIR / "shap" / "model_key_shap_importance.csv"
shap_summary = pd.read_csv(model_key_shap_path) if model_key_shap_path.exists() else pd.DataFrame()
if not shap_summary.empty:
    shap_summary = shap_summary[shap_summary["class_name"] != "__overall__"].copy()

feature_summary.to_csv(PROFILES_DIR / "model_feature_summary.csv", index=False)
effect_sizes.to_csv(PROFILES_DIR / "model_effect_sizes.csv", index=False)
top_features.to_csv(PROFILES_DIR / "model_top_features.csv", index=False)
sensitivity.to_csv(PROFILES_DIR / "language_paraphrase_sensitivity.csv", index=False)
write_profile_markdown(top_features, shap_summary)

top_features

,model_key,feature,model_mean,other_models_mean,effect_size,abs_effect_size,direction
0,gemini-flash-latest,text_punctuation_density,0.066952,0.032513,1.903455,1.903455,higher
1,gemini-flash-latest,text_newline_count,39.967302,16.626389,1.343023,1.343023,higher
2,gemini-flash-latest,text_paragraph_count,12.702997,6.141204,1.266899,1.266899,higher
3,gemini-flash-latest,text_sentence_count,38.682561,19.282870,1.215893,1.215893,higher
4,gemini-flash-latest,text_punctuation_count,232.746594,83.812963,1.052327,1.052327,higher
5,gemini-flash-latest,text_repeated_trigram_ratio,0.012880,0.073568,-0.766215,0.766215,lower
6,gemini-flash-latest,text_repeated_bigram_ratio,0.059204,0.129538,-0.722433,0.722433,lower
7,gemini-flash-latest,text_negation_word_count,4.031335,1.804630,0.697187,0.697187,higher
8,llama-3.1-8b-groq,text_hapax_ratio,0.352391,0.542057,-1.009808,1.009808,lower
9,llama-3.1-8b-groq,text_repeated_bigram_ratio,0.189465,0.085944,0.944921,0.944921,higher


## Dashboard

After running this notebook, launch the dashboard from the project root:

```powershell
make dashboard
```

The dashboard reads `reports/xai`, `reports/style_profiles`, and the final response CSV files.